In [1]:
import gseapy as gp
from lincs_gsnn.explain.infer_edge_weights import infer_edge_weights
import torch 
import pandas as pd 
from lincs_gsnn.explain.viz import annotate_edges, get_drug_edges, make_subgraph
from pypath.utils import mapping

from matplotlib import pyplot as plt 
import numpy as np 

from lincs_gsnn.models.ODEFunc import ODEFunc

from torchdiffeq import odeint

from scipy.stats import hypergeom
from lincs_gsnn.data.TrajDataset import TrajDataset
from lincs_gsnn.data.DXDTDataset import DXDTDataset
from torch.utils.data import DataLoader

from gsnn.interpret.IGExplainer import IGExplainer
from gsnn.interpret.GSNNExplainer import GSNNExplainer

import seaborn as sbn 

from gsnn.optim.OutputEdgeInferer import OutputEdgeInferer

%load_ext autoreload
%autoreload 2

In [2]:
root1 = '../../workflow_outputs/lincs-gsnn'
root2 = '../../workflow_outputs/lincs-traj'

data = torch.load(f'{root1}/default/bionetwork/bionetwork.pt', weights_only=False)
model = torch.load(f'{root1}/default/pretrain/pretrained_model.pt', weights_only=False).eval()
dxdt_scale = torch.load(f'{root1}/default/pretrain/dxdt_scale.pt', weights_only=False).item()
x_names = pd.read_csv(f'{root2}/runs/exp/default_v02/output/predict_grid/gene_names.csv')['gene_names'].values.astype(str)
dxdt_meta = pd.read_csv(f'{root2}/runs/exp/default_v02/output/predict_grid/dxdt_meta.csv')
x_meta = pd.read_csv(f'{root2}/runs/exp/default_v02/output/predict_grid/pred_meta.csv')

valid_drugs = [x.split('DRUG__')[1] for x in data.node_names_dict['input'] if 'DRUG__' in x]

dxdt_meta = dxdt_meta[dxdt_meta['pert_id'].isin(valid_drugs)] 
x_meta = x_meta[x_meta['pert_id'].isin(valid_drugs)] 

In [3]:
cell_line = 'HME1'

dxdt_meta = dxdt_meta[dxdt_meta['cell_iname'] == cell_line]

In [4]:
dxdt_dir = f'{root2}/runs/exp/default_v02/output/predict_grid/dxdt'

In [5]:
batch_size = 128

train_ids = dxdt_meta.sample(frac=0.8).index 
test_ids = dxdt_meta.index.difference(train_ids) 
train_cond = dxdt_meta.loc[train_ids]
test_cond = dxdt_meta.loc[test_ids]

train_dataset = DXDTDataset(train_cond, 
                        input_names=data.node_names_dict['input'], 
                        output_names=data.node_names_dict['output'], 
                        src_names=x_names, 
                        obs_dir=dxdt_dir, 
                        scale=dxdt_scale) 

test_dataset = DXDTDataset(test_cond, 
                        input_names=data.node_names_dict['input'], 
                        output_names=data.node_names_dict['output'], 
                        src_names=x_names, 
                        obs_dir=dxdt_dir, 
                        scale=dxdt_scale) 

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [6]:
OEI = OutputEdgeInferer(data, model.channels*model.layers, lr=1e-2, wd=1e-4, epochs=3, agg='all', use_batchnorm=True)
res = OEI.fit(train_loader, model, device='cuda')

Fitting OutputEdgeInferer on cuda...
# parameters:  78631200
epoch 0 loss: 1.112320548958248604907]
epoch 1 loss: 1.100529384613037895508]
epoch 2 loss: 1.10163598457972212395]]


In [7]:
torch.cuda.empty_cache()

In [8]:
res  = OEI.evaluate(dataloader=test_loader, model=model, device='cuda', verbose=True)

Evaluating OutputEdgeInferer on cuda...


In [10]:
res.sort_values('r2_gain', ascending=False).head(10)

,func_node,output_node,mse,r2,r,has_edge,p_value,snr,l1_norm,l2_norm,...,model_r2,model_r,model_mse,r2_gain,r_gain,mse_gain,q_value,snr_rank,sparsity_rank,within_output_rank
0,RNA__GOLT1B,GENE__ATP6V0B,0.226702,0.363247,0.743053,False,0.0,0.149706,0.210142,0.027805,...,0.000000,0.037979,0.359129,0.363247,0.705075,-0.132427,0.0,2,89,57
1,RNA__TLK2,GENE__EPHA3,0.283254,0.362594,0.722643,False,0.0,0.202918,0.256134,0.033434,...,0.000000,0.129017,0.445922,0.362594,0.593626,-0.162667,0.0,2,89,26
2,RNA__VDAC1,GENE__EPHA3,0.283372,0.362331,0.717714,False,0.0,0.208050,0.261209,0.034973,...,0.000000,0.129017,0.445922,0.362331,0.588697,-0.162550,0.0,1,88,25
3,RNA__GOLT1B,GENE__KIAA0100,0.473399,0.379486,0.778335,False,0.0,0.165944,0.320882,0.042701,...,0.020478,0.188298,0.747291,0.359009,0.590037,-0.273893,0.0,2,74,40
4,RNA__SLC25A14,GENE__KIAA0100,0.474017,0.378676,0.827477,False,0.0,0.135112,0.321678,0.044336,...,0.020478,0.188298,0.747291,0.358198,0.639179,-0.273274,0.0,6,78,44
5,RNA__DENND2D,GENE__COPS7A,0.527975,0.357461,0.808963,False,0.0,0.194551,0.349516,0.046488,...,0.000000,0.099491,0.846142,0.357461,0.709472,-0.318167,0.0,1,77,39
6,RNA__SLC25A14,GENE__RNF167,0.171659,0.353093,0.772197,False,0.0,0.120659,0.179302,0.026050,...,0.000000,0.017030,0.270607,0.353093,0.755167,-0.098948,0.0,2,83,49
7,RNA__SLC25A14,GENE__PHKG2,0.228606,0.351860,0.770201,False,0.0,0.120550,0.205233,0.028996,...,0.000000,-0.088768,0.360937,0.351860,0.858969,-0.132331,0.0,1,81,69
8,RNA__CDK19,GENE__SENP6,0.190936,0.351732,0.808685,False,0.0,0.129565,0.200222,0.027452,...,0.000000,-0.165933,0.315140,0.351732,0.974619,-0.124204,0.0,5,95,88
9,RNA__EIF4G1,GENE__ATP6V0B,0.231355,0.350178,0.743468,False,0.0,0.133680,0.197196,0.027184,...,0.000000,0.037979,0.359129,0.350178,0.705489,-0.127775,0.0,8,95,63
